# [GPU 실습] 미세먼지 기사 제목 수천 개를 '뜻'으로 묶기
문장을 숫자 목록(임베딩)으로 바꾸면, 뜻이 비슷한 제목끼리 가까워져요. 가까운 것끼리 묶으면 **보도가 어떤 갈래로 나뉘는지** 한눈에 보여요.

준비: 1번 노트북에서 쓴 빅카인즈 파일(`NewsResult_*.xlsx`)을 같은 폴더에 · GPU가 없으면 CPU로도 돌아가요(느릴 뿐)

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
# Colab용 설치 칸. VS Code는 README 순서대로(GPU용 torch 먼저) 설치했다면 건너뛰기
#   ⚠️ torch 없이 이 칸부터 실행하면 CPU용 torch가 깔려 GPU를 못 써요
if IN_COLAB:
    get_ipython().run_line_magic('pip', 'install -q sentence-transformers')

In [ ]:
import glob, time
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('사용 장치:', DEVICE)

files = sorted(glob.glob('*NewsResult_*.xlsx'))
news = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)
news = news.dropna(subset=['제목']).drop_duplicates(subset=['제목'])     # 같은 제목(통신사 전재) 빼기
titles = news['제목'].tolist()
print(len(titles), '개 제목')

## 1. 한국어 문장 모델 불러오기 + CPU vs GPU 속도 비교

In [ ]:
model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=DEVICE)   # 처음 한 번 내려받기(약 400MB)

sample = titles[:500]
for dev in (['cpu', 'cuda'] if DEVICE == 'cuda' else ['cpu']):
    model.to(dev)
    t = time.time(); model.encode(sample, batch_size=64)
    print(f'{dev}: 제목 500개 → {time.time() - t:.1f}초')
model.to(DEVICE)

## 2. 전체 제목을 숫자로 바꾸고, 비슷한 것끼리 묶기

In [ ]:
emb = model.encode(titles, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
print(emb.shape)      # (제목 수, 768) — 제목 하나가 숫자 768개

from sklearn.cluster import KMeans
K = 6                 # 묶음 수 — 4~8로 바꿔 보기
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(emb)
news['묶음'] = km.labels_

## 3. 묶음마다 대표 제목 읽기 → 이름 붙이기 (사람의 일)

In [ ]:
for k in range(K):
    idx = np.where(km.labels_ == k)[0]
    dist = np.linalg.norm(emb[idx] - km.cluster_centers_[k], axis=1)
    print(f'■ 묶음 {k} ({len(idx)}건)')
    for i in idx[dist.argsort()[:5]]:          # 묶음 한가운데에 가까운 제목 5개
        print('   ', titles[i])

In [ ]:
# 대표 제목을 읽고 이름을 붙이세요 (예: '예보·날씨', '건강', '중국발', '정책')
names = {0: '묶음0', 1: '묶음1', 2: '묶음2', 3: '묶음3', 4: '묶음4', 5: '묶음5'}
news['갈래'] = news['묶음'].map(names)

## 4. 기사용 차트 — 가장 큰 갈래 하나만 강조

In [ ]:
cnt = news['갈래'].value_counts().sort_values()
colors = ['#E8590C' if i == len(cnt) - 1 else '#B8B2A7' for i in range(len(cnt))]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(cnt.index, cnt.values, color=colors)
ax.bar_label(bars, padding=4)
ax.set_title(f"미세먼지 보도의 {cnt.values[-1] / cnt.sum():.0%}는 '{cnt.index[-1]}'", loc='left', fontsize=15, fontweight='bold')
fig.text(0.01, 0.01, f'출처: 빅카인즈 · 2022-04~12 · 제목 {cnt.sum():,}건(중복 제외)', fontsize=9, color='gray')
ax.set_xlabel('기사 수(건)'); plt.show()

### 읽을 때 주의
- 묶음은 컴퓨터가 **비슷한 정도**로 나눈 것이지 정답 분류가 아니에요. 이름은 사람이 대표 제목을 읽고 붙여요.
- K를 바꾸면 갈래도 바뀌어요. 기사에 쓸 때는 "제목을 뜻으로 묶어 보니"처럼 **방법을 밝혀요.**